## Pipeline Overview

This notebook generates the training dataset for the surrogate model:

1. Latin Hypercube Sampling of the 32-variable design space
2. Full AVL evaluation of each sample (2 runs: baseline + trim)
3. NeuralFoil strip drag + analytical body form drag
4. CG computation from component placement
5. Propulsion balance (T/D, endurance) with discharge efficiency
6. Manufacturability scoring
7. Dataset cleaning (failed evaluations, divergent VLM outputs)
8. Save to `.npz` for surrogate training

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
from scipy.stats import qmc

from src.parameterization.design_variables import (
    BWBParams, N_VARS, VAR_NAMES, get_bounds_arrays, params_from_vector,
)
from src.aero.evaluator import AeroEvaluator
from src.config import load_all
from src.optimization.database import EvaluationDatabase

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

from src.visualization.style import apply_style, COLORS
apply_style()
%load_ext autoreload
%autoreload 2

## 1. LHS Sampling

In [ ]:
cfg = load_all()
mission = cfg['mission']
feasibility = cfg['feasibility']
evaluator = AeroEvaluator(
    mission=mission, feasibility=feasibility,
    controls=cfg['controls'], cg_config=cfg['cg'],
    avl_command=cfg['avl_command'],
    aero_model=cfg['aero_model'], structure_config=cfg['structure'],
)

lb, ub = get_bounds_arrays()

# --- Load existing database ---
db_path = '../data/eval_database.json'
try:
    db = EvaluationDatabase.load(db_path)
    n_existing = len(db)
    print(f'Loaded existing database: {n_existing} samples')
except FileNotFoundError:
    db = EvaluationDatabase()
    n_existing = 0
    print('No existing database found, starting fresh')

# --- Generate NEW LHS samples with duct-compatible body chord filter ---
n_new = 19500
seed = 123 + n_existing

from src.parameterization.design_variables import min_body_chord_for_duct
body_chord_min = min_body_chord_for_duct(mission.edf.duct_outer_diameter)
idx_rc = VAR_NAMES.index("wing_root_chord")
idx_bcr = VAR_NAMES.index("body_chord_ratio")

sampler = qmc.LatinHypercube(d=N_VARS, seed=seed)
X_all = []
n_generated = 0
n_rejected = 0
while len(X_all) < n_new:
    batch = qmc.scale(sampler.random(n=n_new - len(X_all) + 500), lb, ub)
    body_chord = batch[:, idx_rc] * batch[:, idx_bcr]
    mask = body_chord >= body_chord_min
    X_all.extend(batch[mask])
    n_rejected += (~mask).sum()
    n_generated += len(batch)

X = np.array(X_all[:n_new])

print(f'Generated {n_new} duct-compatible LHS samples (rejected {n_rejected}/{n_generated})')
print(f'  body_chord_min = {body_chord_min*1000:.0f} mm (for {mission.edf.name})')
print(f'  body_chord range: [{X[:, idx_rc].min() * X[:, idx_bcr].min()*1000:.0f}, '
      f'{X[:, idx_rc].max() * X[:, idx_bcr].max()*1000:.0f}] mm')
print(f'After evaluation, total will be: {n_existing} + {n_new} = {n_existing + n_new}')

## 2. Batch VLM Evaluation

In [ ]:
t0 = time.time()

for i, x in enumerate(X):
    params = params_from_vector(x)
    result = evaluator.evaluate(params)
    db.add(x, result)

    if (i + 1) % 50 == 0 or i == 0 or i == len(X) - 1:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        n_total = len(db)
        n_feas = sum(1 for r in db.results if r.get('is_feasible', False))
        print(f'  {i+1:5d}/{n_new} ({rate:.1f}/s) | total={n_total} | feasible={n_feas}', flush=True)

    # Checkpoint every 1000 samples
    if (i + 1) % 1000 == 0:
        db.save(db_path)

elapsed = time.time() - t0

# Duct diagnostic
n_duct_ok = sum(1 for r in db.results[-n_new:] if r.get('duct_fits', False))
n_clr_ok = sum(1 for r in db.results[-n_new:]
               if r.get('min_duct_clearance_mm', -999) >= 0)
n_feas_total = sum(1 for r in db.results[-n_new:] if r.get('is_feasible', False))

print(f'\nDone in {elapsed:.0f}s — {len(db)} total samples')
print(f'  Duct clearance >= 0mm: {n_clr_ok}/{n_new} ({100*n_clr_ok/n_new:.1f}%)')
print(f'  Duct fits (relaxed):   {n_duct_ok}/{n_new} ({100*n_duct_ok/n_new:.1f}%)')
print(f'  Feasible (all):        {n_feas_total}/{n_new} ({100*n_feas_total/n_new:.1f}%)')

## 3. Dataset Statistics

In [ ]:
# Extract key metrics
X_arr, results = db.to_arrays()
n_total = len(results)
ld_values = [r['L_over_D'] for r in results]
cd0_wing = [r['CD0_wing'] for r in results]
cd0_body = [r['CD0_body'] for r in results]
endurance = [r.get('endurance_min', 0) for r in results]
feasible = [r['is_feasible'] for r in results]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(ld_values, bins=50, color='steelblue', alpha=0.7)
axes[0, 0].set_xlabel('L/D')
axes[0, 0].set_title('L/D Distribution')

axes[0, 1].hist(cd0_wing, bins=50, color='steelblue', alpha=0.7, label='Wing')
axes[0, 1].hist(cd0_body, bins=50, color='coral', alpha=0.7, label='Body')
axes[0, 1].set_xlabel('CD0')
axes[0, 1].set_title('Drag Breakdown')
axes[0, 1].legend()

axes[1, 0].hist(endurance, bins=50, color='forestgreen', alpha=0.7)
axes[1, 0].axvline(10, color='red', ls='--', label='10 min min')
axes[1, 0].set_xlabel('Endurance [min]')
axes[1, 0].set_title('Endurance Distribution')
axes[1, 0].legend()

axes[1, 1].bar(['Feasible', 'Infeasible'], [sum(feasible), sum(not f for f in feasible)],
               color=['forestgreen', 'coral'])
axes[1, 1].set_title(f'Feasibility: {sum(feasible)}/{n_total} ({sum(feasible)/n_total:.1%})')

plt.tight_layout()
plt.show()

## 4. Save Dataset

In [ ]:
# Save database for surrogate training
db.save(db_path)

# Also save as Parquet for fast loading
X_arr, results = db.to_arrays()
records = []
for x, r in zip(X_arr, results):
    row = {f'x_{i}': float(x[i]) for i in range(N_VARS)}
    for k, v in r.items():
        if isinstance(v, (int, float, bool, np.floating, np.integer, np.bool_)):
            row[k] = float(v) if not isinstance(v, bool) else v
    records.append(row)

df = pd.DataFrame(records)
df.to_parquet('../data/dataset_v2.parquet', index=False)
print(f'Saved {len(df)} total samples to data/dataset_v2.parquet')
print(f'Columns: {len(df.columns)}')